In [1]:
import os
%pwd

'd:\\MLops Projects\\Datascience_Project\\research'

In [2]:
os.chdir("../")
%pwd

'd:\\MLops Projects\\Datascience_Project'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path 
    

In [4]:
from src.DataScience_Project.constants import *
from src.DataScience_Project.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [6]:
import os
import urllib.request as request
from src.DataScience_Project import logger
import zipfile

In [7]:
import os
import zipfile
import urllib.request as request
from src.DataScience_Project import logger

class DataIngestion:

    def __init__(self, config):
        self.config = config

    # downloading the zip file
    def download_file(self):

        if not os.path.exists(self.config.local_data_file):

            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )

            logger.info(
                f"{filename} downloaded with following info: \n{headers}"
            )

        else:
            logger.info("File already exists")

    # extract the zip file
    def extract_zip_file(self):

        """
        Extract the zip file into the data directory
        """

        unzip_path = self.config.unzip_dir

        os.makedirs(unzip_path, exist_ok=True)

        with zipfile.ZipFile(
            self.config.local_data_file,
            'r'
        ) as zip_ref:

            zip_ref.extractall(unzip_path)